# 79 · Published human vs mouse PT programs along the package pseudospace

With two specimens per species, a 1,513-set discovery screen cannot carry new pathway claims. This notebook goes
hypothesis-driven instead: it takes programs that atlases and microdissection studies already published, fixed
before any look at our data, and asks three questions of each.

| Part | Question |
|---|---|
| (a) Replication | Does our cohort reproduce the published claim? |
| (b) Localization | Where along the PT is the human − mouse difference: which segment, uniform or segment-specific, replicated across the four specimen pairs? |
| (c) Added information | What does that add to the published claim? |

**Programs** (`docs/results/known-programs/programs_hvm.csv`, frozen, sha256 asserted):
- **Species family S:** six CellSpectra Supp Table 4 blocks, human-high or mouse-high in PTS1, PTS2 or PTS3.
- **Zonation family Z:** S1/S2/S3 programs of mouse microdissection (Chen 2021) and of human within-PT markers
  (Hinze 2022).
- **Secondary:** other zonation sources.

**Protocol.** `docs/results/pt-known-programs.md` (commit b90a682), frozen before this notebook ran, with the
coordinator's decisions O1–O8.

**What bounds the reading.**
- Notebook 72's C14 is not met, so localization is claimed at segment level only and within-segment shape is
  descriptive.
- The between-segment species contrast does not depend on the two probe panels; the level does.

**Cohort and scope.** Two male control mice and two cortex sections of one male human donor. Specimens are the
replicates, and everything is descriptive. Nothing here refits a gene model.

## 1 · Setup

In [ ]:
import argparse
import hashlib
import json
import os
import sys
import time
from pathlib import Path

for _var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(_var, os.environ.get('PSEUDOSPACE_N_JOBS', '24'))   # thread cap; never in a cache key
NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
EXPORT = results_root / 'package_pseudospace' / 'hvm' / 'pt_units.h5ad'
upstream13 = results_root / 'minimal_pt_scfates'                  # the ortholog map
T71 = results_root / 'pt_pathways_package_pseudospace'            # read only: tables and stage cache
T72 = results_root / 'pt_package_coordinate_validation'
T75 = results_root / 'pt_package_pathway_methods'
nb38 = results_root / 'pt_literature_deep'
PROGRAM_DIR = project / 'docs' / 'results' / 'known-programs'      # committed, public-literature derived
ENSEMBL = data_root / 'external' / 'ensembl_r110_human_transcripts' / 'Homo_sapiens.GRCh38.cdna.all.fa.gz'
output = results_root / 'pt_known_species_programs'
tables, figures, cache = output / 'tables', output / 'figures', output / 'stage_cache'
for folder in (tables, figures, cache):
    folder.mkdir(parents=True, exist_ok=True)
DS = 'human_vs_mouse'                                             # notebook 71's dataset name
PARTITION_FILES = {'condition': 'species', 'relabeling 1': 'swap_HUK1_COR1_Ctrl1A2', 'relabeling 2': 'swap_HUK1_COR1_Ctrl1A4'}
PROGRAM_SHA = {'programs_hvm.csv': 'be61b8183c51d798ee33314e6226d5d8dd269188b82eb60ae65c888a355db79e',
               'programs_hvm_members.csv': '2a3889075ca4130b102661409aee1390f1dd1f936a4171c9c9b7b0a7365299bf'}
ENSEMBL_SHA = '1881e80298f19207159b60bf1eb79b0c5fe34eb38c24c5b4e796a35f016c1e7d'
REQUIRED = [EXPORT, upstream13 / 'ortholog_map_used.csv', ENSEMBL, nb38 / 'probe_counts.csv',
            *[PROGRAM_DIR / f for f in PROGRAM_SHA],
            *[T71 / 'tables' / f for f in ('inputs.csv', f'universe_{DS}.csv', f'gene_sets_{DS}.json', f'gene_statistics_{DS}.csv',
                                           *[f'pathway_tests_{DS}_cameraPR_{v}.csv' for v in PARTITION_FILES.values()])],
            T72 / 'tables' / 'transitions.csv',
            *[T75 / 'tables' / f for f in (f'gene_statistics_segment_{DS}.csv', f'pathway_tests_segment_{DS}.csv',
                                           f'deseq2_genes_{DS}.csv')],
            *[data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{lib}.json' for lib in ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 70–72 and 75 first)')

NOTEBOOK_LOGIC_VERSION = '79.known_programs.1'          # Bump when a cached calculation changes.
FINGERPRINT = 'efafc9ac2e99'                            # confirmed hvm pass-1 labels (notebook 70)
ALPHA, CAMERA_COR, CALIBRATION_MAX = .05, .01, .075     # project set rule; cameraPR; 75's swap rule
EFFECT_MIN, RATIO_MAX = .25, .25                         # 76's O5 thresholds (log2; relabeled ratio)
DF1, DF2 = 6, 12                                         # notebook 71's positional F-test (guard)
MIN_MEMBERS, N_BINS, MAX_ROWS, LENGTH_BINS = 5, 20, 40, 10
FLOOR = .5                                               # pseudobulk floor (71's 0.5 count)
SEGMENTS = ('S1', 'S2', 'S3')
SCOPES = {'S1': ['S1'], 'S2': ['S2'], 'S3': ['S3'], 'PT': None}
PARTS = ['condition', 'relabeling 1', 'relabeling 2']
NAMES = ['Ctrl1A2', 'Ctrl1A4', 'HUK1_COR1', 'HUK1_MED1']
MOUSE, HUMAN = NAMES[:2], NAMES[2:]
CASE, REF = ('HUK1_COR1', 'HUK1_MED1'), ('Ctrl1A2', 'Ctrl1A4')       # relabeling 1 = (c1, r1), relabeling 2 = (c1, r2)
PAIRS = [f'{c}−{r}' for c in CASE for r in REF]
LABEL_ORIENTATION = {'Cyp2e1', 'Cyp7b1', 'Gatm', 'Slc7a13', 'Slc13a3', 'Slc22a6', 'Slc5a2', 'Slc22a7', 'Slc5a12',
                     'Aqp2', 'Aqp4', 'Atp6v0d2', 'Rhcg', 'Slc14a2', 'Wnt7b'}  # protocol R3 (72's exclusion)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
MOUSE_COLOR, HUMAN_COLOR, RELABEL_COLOR = '#0072B2', '#D55E00', '#9a9a9a'
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5, 'axes.titleweight': 'bold',
    'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.fontsize': 6, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': .6, 'lines.linewidth': 1.0,
    'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 90, 'display.max_rows', 120)
READING, GUARDS = [], []


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')
    plt.show()
    plt.close(fig)


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


print('Results folder:', output, '· logic', NOTEBOOK_LOGIC_VERSION)
READING.append(f"Logic version {NOTEBOOK_LOGIC_VERSION!r}, by the coordinator's phase-2 instruction (the protocol named "
               "'79.known_species_programs.1').")

## 2 · Inputs and guards

The inputs are built as notebook 71 builds them (`package_inputs.hvm_inputs`; the 13 ortholog-map `mt-` genes are
out of the library and the universe). Guards (not cached):
- **G1:** the export contract and the confirmed pass-1 fingerprint.
- **G2:** 71's universe (11,055 genes) and units (11,383).
- **G3:** 71's cached condition model reproduces its T_spatial and moderated-F z; cameraPR on 71's z reproduces its
  set tests.
- **G4:** 75's segment table covers the universe; cameraPR on its z reproduces 75's set tests.
- **G5:** the program files match their frozen sha256; members lie in the universe; none is a label or orientation
  gene.
- **G6:** the logistic transitions reproduce 72's.
- **G7:** the Ensembl FASTA matches its sha256.

In [ ]:
from pseudospace.continuous_descriptors import logistic_transitions
from pseudospace.discrete_comparators import swap_rule
from pseudospace.package_inputs import hvm_inputs, read_pt_units
from pseudospace.pathway_pipelines import camera_pr, common_grid, moderated_f, partition_groups
from pseudospace import program_localization as pl
from pseudospace import species_calls as sc

t0 = time.perf_counter()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
MT = sorted(g for g in orthologs.mouse_symbol.astype(str) if g.startswith('mt-'))
assert len(MT) == 13, MT
UNITS = read_pt_units(EXPORT, 'hvm')                                                       # G1
PROV = UNITS.uns['provenance']
assert str(PROV['notebook_logic_version']).startswith('70.'), PROV['notebook_logic_version']
assert PROV['fingerprints']['pass1']['membership_sha1'] == FINGERPRINT, PROV['fingerprints']['pass1']
panels = PROV['config']['marker_panels_yaml']
label_orientation = ({str(g) for s in ('PT-S1', 'PT-S2', 'PT-S3') for g in panels['label_panels'][s]}
                     | {str(g) for side in ('early', 'late') for g in panels['axis_anchors'][side]})
assert label_orientation == LABEL_ORIENTATION, label_orientation ^ LABEL_ORIENTATION
INP = hvm_inputs(UNITS, data_root, orthologs, exclude_genes=MT)
GENES = list(INP['genes'])
SETS = INP['gene_sets']
universe71 = pd.read_csv(T71 / 'tables' / f'universe_{DS}.csv')
universe71 = universe71[universe71.eligible.astype(bool)]
assert GENES == universe71.gene.astype(str).tolist(), 'G2: the universe differs from notebook 71'
assert SETS == json.loads((T71 / 'tables' / f'gene_sets_{DS}.json').read_text()), 'G2: the gene sets differ from notebook 71'
assert int(pd.read_csv(T71 / 'tables' / 'inputs.csv', index_col=0).loc['units in the common support', 'Human vs mouse PT']) == len(INP['position'])
POSITION = np.asarray(INP['position'], float)
SPECIMEN = INP['specimen'].astype(str)
SEGMENT = pd.Series(INP['segment']).str.replace('PT-', '', regex=False).to_numpy()
COUNTS = INP['counts'].tocsr()
LIBRARY = np.asarray(INP['library'], float)                          # mt-free (the NB offset)
LOG_MEAN_COUNT = np.log(np.asarray(COUNTS.mean(axis=0)).ravel())     # 71's squeezeVar covariate
labels = list(partition_groups(SPECIMEN, INP['case']))
assert labels == ['species', 'swap:HUK1_COR1+Ctrl1A2', 'swap:HUK1_COR1+Ctrl1A4'], labels
GUARDS.append(f'G1/G2: export contract; pass-1 fingerprint {FINGERPRINT}; universe ({len(GENES):,}), sets ({len(SETS):,}) and '
              f'units ({len(POSITION):,}) equal notebook 71; the label/orientation genes equal the protocol\'s 15')


def stored(folder, stage):
    """A notebook's newest cached payload of one stage (read only); raises when its cache was purged."""
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        raise FileNotFoundError(f'{folder.name}: no cached {stage}')
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


STAT71 = pd.read_csv(T71 / 'tables' / f'gene_statistics_{DS}.csv', header=[0, 1], index_col=0)
assert STAT71.index.astype(str).tolist() == GENES
MODELS = {part: stored(T71, f'gene_model_{DS}_{f}') for part, f in PARTITION_FILES.items()}
np.testing.assert_allclose(MODELS['condition']['stat|T_spatial'], STAT71['condition'].T_spatial.to_numpy(), rtol=1e-12, atol=0)
again = moderated_f(STAT71['condition'].T_spatial, DF1, STAT71['condition'].LR_spec.to_numpy(), DF2,
                    covariate=LOG_MEAN_COUNT, robust=False)
np.testing.assert_allclose(again.z.to_numpy(), STAT71['condition'].z.to_numpy(), rtol=1e-9, atol=1e-9)
GRID = MODELS['condition']['grid']
np.testing.assert_allclose(GRID, common_grid(POSITION, SPECIMEN, 101), rtol=0, atol=1e-12)
Z_POS = {part: STAT71[part].z.astype(float) for part in PARTS}
TESTS71 = {part: pd.read_csv(T71 / 'tables' / f'pathway_tests_{DS}_cameraPR_{f}.csv').set_index('pathway_id')
           for part, f in PARTITION_FILES.items()}
GUARDS.append("G3: notebook 71's cached condition model reproduces its T_spatial and moderated-F z")

SEG75 = pd.read_csv(T75 / 'tables' / f'gene_statistics_segment_{DS}.csv', header=[0, 1], index_col=0)
assert SEG75.index.astype(str).tolist() == GENES
Z_SEG = {part: SEG75[part].z.astype(float) for part in PARTS}
SEGTESTS75 = pd.read_csv(T75 / 'tables' / f'pathway_tests_segment_{DS}.csv', header=[0, 1], index_col=0)

PROGRAMS_ALL, META = pl.read_programs(PROGRAM_DIR / 'programs_hvm.csv', GENES, LABEL_ORIENTATION,
                                      sha256=PROGRAM_SHA['programs_hvm.csv'])
assert sha256(PROGRAM_DIR / 'programs_hvm_members.csv') == PROGRAM_SHA['programs_hvm_members.csv']
MEMBERS_TABLE = pd.read_csv(PROGRAM_DIR / 'programs_hvm_members.csv')
assert all(MEMBERS_TABLE[MEMBERS_TABLE.program_id.eq(p)].gene.tolist() == v for p, v in PROGRAMS_ALL.items())
GUARDS.append(f'G5: program files match their frozen sha256; {len(PROGRAMS_ALL)} programs, every member in the universe, '
              'no label/orientation gene')

TRANSITIONS = pd.read_csv(T72 / 'tables' / 'transitions.csv')
TRANSITIONS = TRANSITIONS[TRANSITIONS.dataset.eq('hvm')].reset_index(drop=True)
again_t = logistic_transitions(POSITION, INP['segment'], SPECIMEN).merge(TRANSITIONS, on=['specimen', 'transition'])
assert len(again_t) == 8 and np.allclose(again_t.position_x, again_t.position_y, atol=1e-6)   # G6
CUTS = sc.mean_transitions(TRANSITIONS, NAMES)
S_GRID = sc.to_segment_units(GRID, CUTS)
GUARDS.append(f"G6: logistic transitions reproduce 72's; pooled cuts S1→S2 {CUTS[0]:.3f}, S2→S3 {CUTS[1]:.3f}")
assert sha256(ENSEMBL) == ENSEMBL_SHA                                                        # G7
GUARDS.append('G7: the Ensembl 110 cDNA FASTA matches its sha256')
print('\n'.join('· ' + g for g in GUARDS))
print(f'Inputs {time.perf_counter() - t0:.0f} s · {len(POSITION):,} units · {len(GENES):,} genes · {len(SETS):,} sets')

## 3 · The frozen programs

Confirmatory: family S (six CellSpectra blocks) and family Z (six zonation programs). Secondary: the other zonation
sources (nominal p, never pooled into a family). Every member is a one-to-one ortholog in 71's universe and none is a
label or orientation gene (protocol R1–R5).

In [ ]:
META = META.copy()
META['n_used'] = [len(PROGRAMS_ALL[p]) for p in META.index]
CONFIRMATORY = {fam: [p for p in META.index if META.loc[p, 'role'] == 'confirmatory' and META.loc[p, 'family'] == fam]
                for fam in ('species', 'zonation')}
SECONDARY = [p for p in META.index if META.loc[p, 'role'] == 'secondary']
assert len(CONFIRMATORY['species']) == 6 and len(CONFIRMATORY['zonation']) == 6, CONFIRMATORY
SOURCE_SPECIES = {p: ('human' if '(human)' in str(META.loc[p, 'direction']) else 'mouse') for p in META.index
                  if META.loc[p, 'family'] == 'zonation'}
CLAIM_SIGN = {p: (1 if str(META.loc[p, 'direction']).startswith('human-high') else -1)
              for p in CONFIRMATORY['species']}                       # human - mouse sign the paper claims
display(META[['family', 'role', 'source', 'direction', 'segment', 'n_listed', 'n_non_one_to_one', 'n_unmapped_symbol',
              'n_not_in_universe', 'n_label_orientation_excluded', 'n_members']])

## 4 · Per-gene statistics

**Coordinate-free (new, protocol 3.2–3.4).** These are computed on the pseudobulks of 71's analysed units.
- **Pseudobulks:** log2((Σ counts + 0.5) / Σ mt-free library) per specimen × scope (S1, S2, S3, PT).
- **Contrasts,** each centred by its median over the universe genes:
  - the group effect (human − mouse);
  - the 4 human × mouse pairs;
  - the relabeled effects.
- **Moderated level t** (3.3): squeezeVar with the log mean count per unit, robust FALSE.
- **Zonation (3.4):** each species' segment-vs-other-segments contrast, with its own moderated t.

**Reused.**
- The segment F-test z (75), the positional F-test z (71), and 71's curves.
- The DESeq2 Wald stat per scope (75), as the conventional comparator.

In [ ]:
def pseudobulks(counts, library, specimen, segment):
    return pl.segment_pseudobulk(counts, library, specimen, segment, SCOPES, genes=GENES, floor=FLOOR)


def level_statistics(rates, covariate):
    """{(scope, partition): moderated t frame} and the pair contrasts {scope: frame pairs}."""
    stats, pairs = {}, {}
    for scope in rates:
        for part in PARTS:
            eff = pl.partition_effects(rates[scope], CASE, REF, part)
            stats[(scope, part)] = pl.moderated_t(eff.effect, eff.s2, float(eff.df.iloc[0]), covariate=covariate)
        pairs[scope] = pd.DataFrame(pl.pair_contrasts(rates[scope], CASE, REF))
    return stats, pairs


def zonation_statistics(rates, covariate):
    """{(species, segment): moderated t frame with the per-specimen contrasts}."""
    out = {}
    for species, specimens in (('mouse', MOUSE), ('human', HUMAN)):
        for seg in SEGMENTS:
            con = pl.segment_contrast({s: rates[s] for s in SEGMENTS}, specimens, seg)
            t = pl.moderated_t(con.effect, con.s2, 1., covariate=covariate)
            out[(species, seg)] = t.join(con[specimens])
    return out


t0 = time.perf_counter()
RATES = pseudobulks(COUNTS, LIBRARY, SPECIMEN, SEGMENT)
LEVEL, PAIR_CON = level_statistics(RATES, LOG_MEAN_COUNT)
ZON = zonation_statistics(RATES, LOG_MEAN_COUNT)
DESEQ = pd.read_csv(T75 / 'tables' / f'deseq2_genes_{DS}.csv')
DESEQ_PART = {'condition': 'species', 'relabeling 1': 'swap:HUK1_COR1+Ctrl1A2', 'relabeling 2': 'swap:HUK1_COR1+Ctrl1A4'}
DESEQ_SCOPE = {'S1': 'S1', 'S2': 'S2', 'S3': 'S3', 'PT': 'whole_PT'}


def deseq_stat(part, scope):
    block = DESEQ[DESEQ.partition.eq(DESEQ_PART[part]) & DESEQ.scope.eq(DESEQ_SCOPE[scope])].set_index('gene').stat
    return block.dropna()


probes = pd.read_csv(nb38 / 'probe_counts.csv').drop_duplicates('mouse_symbol').set_index('mouse_symbol').reindex(GENES)
EQUAL_PROBE = [g for g in GENES if pd.notna(probes.loc[g, 'mouse_probes']) and probes.loc[g, 'mouse_probes'] == probes.loc[g, 'human_probes']]
human_symbol = UNITS.var.human_symbol.reindex(GENES).astype(str)
SPAN = pl.gene_span(ENSEMBL, human_symbol.to_numpy())
SPAN.index = GENES
LENGTH_STRATA = pd.qcut(np.log10(SPAN), LENGTH_BINS, labels=False)
level_summary = pd.DataFrame([{'scope': s, 'partition': p, 'd0 (median)': float(LEVEL[(s, p)].d0.median()),
                               'genes p ≤ 0.05': int((LEVEL[(s, p)].p <= .05).sum())} for s in SCOPES for p in PARTS])
display(level_summary)
print(f'Per-gene statistics {time.perf_counter() - t0:.0f} s · equal-probe genes {len(EQUAL_PROBE):,} · '
      f'genes with a span {int(SPAN.notna().sum()):,} of {len(GENES):,}')
READING.append("Equal probe counts follow notebook 77's code (mouse_probes == human_probes), which the protocol calls "
               "'probe_balance equal'.")

## 5 · Calibration of each statistic on the library sets (protocol 4a, 6)

cameraPR (inter.gene.cor 0.01) of every statistic on 71's 1,513 sets, for the condition and both relabelings. 75's
swap rule then applies:
- **Level t** (two-sided): a call is FDR ≤ 0.05 in either direction, and the calibration p is the relabelings'
  two-sided p.
- **Segment and positional F z:** a call is Up with FDR ≤ 0.05, and the calibration p is the one-sided p.

A failing statistic's program p becomes the empirical p against the pooled relabeled p (76's rule). Guards: cameraPR
on 75's segment z and on 71's z reproduces their published set tests.

In [ ]:
def calibration_runs(statistics, name, two_sided):
    """cameraPR of a statistic per partition on the library sets; swap-rule verdict and the pooled relabeled null."""
    runs = {part: camera_pr(statistics[part], SETS, inter_gene_cor=CAMERA_COR).set_index('pathway_id') for part in PARTS}
    if two_sided:
        calls = {part: int((runs[part].fdr <= ALPHA).sum()) for part in PARTS}
        null = np.concatenate([runs[p].p.to_numpy(float) for p in PARTS[1:]])
    else:
        calls = {part: int(runs[part].call.sum()) for part in PARTS}
        null = np.concatenate([runs[p].p_one_sided.to_numpy(float) for p in PARTS[1:]])
    verdict = swap_rule(calls['condition'], [calls[p] for p in PARTS[1:]], len(runs['condition']), null)
    row = {'statistic': name, 'tested': len(runs['condition']), **{f'calls {p}': calls[p] for p in PARTS}, **verdict}
    return row, null, runs


t0 = time.perf_counter()
CALIB, NULLS = [], {}
for scope in SCOPES:
    row, NULLS[('level', scope)], _ = calibration_runs({p: LEVEL[(scope, p)].t for p in PARTS}, f'level t · {scope}', True)
    CALIB.append(row)
row, NULLS['z_seg'], runs_seg = calibration_runs(Z_SEG, 'segment F z (75)', False)
CALIB.append(row)
row, NULLS['z_pos'], runs_pos = calibration_runs(Z_POS, 'positional F z (71)', False)
CALIB.append(row)
for part in PARTS:                                                                            # G3/G4 set-test guards
    np.testing.assert_allclose(runs_seg[part].p.reindex(SEGTESTS75.index), SEGTESTS75[part].p, rtol=1e-8, atol=1e-12)
    np.testing.assert_allclose(runs_pos[part].p.reindex(TESTS71[part].index), TESTS71[part].p, rtol=1e-8, atol=1e-12)
GUARDS.append("G3/G4: cameraPR on 71's z and on 75's segment z reproduces their set tests in every partition")
CALIBRATION = pd.DataFrame(CALIB).set_index('statistic')
PASSES = {name: CALIBRATION.loc[name, 'calibration'] == 'pass' for name in CALIBRATION.index}
CALIBRATION.to_csv(tables / 'calibration.csv')
display(CALIBRATION.round(3))
print(f'Calibration {time.perf_counter() - t0:.0f} s')

## 6 · Program tests

Each test is one cameraPR (inter.gene.cor 0.01) of the program against all other universe genes.

**Replication** (family S): the level t at the claim's segment, one-sided in the claimed direction (human − mouse
sign).
- Calibrated when the level statistic fails its swap rule at that scope.
- Specificity: the program's own relabeled one-sided p is above 0.05 in both relabelings.
- Family Z instead uses the within-species t of its own segment (Up): in the source species (the confirmatory
  positive control) and in the other species (the transfer).

**Between segments:** the segment F z (Up). **Along the axis:** the positional F z (Up; a sensitivity).
**Conventional comparator:** the DESeq2 Wald stat at the claim's scope (descriptive).

BH runs within each family × question.

In [ ]:
from scipy.stats import false_discovery_control


def bh(p):
    p = pd.Series(p, dtype=float)
    out = pd.Series(np.nan, index=p.index)
    ok = p.notna()
    if ok.any():
        out[ok] = false_discovery_control(p[ok].to_numpy())
    return out


def with_calibration(res, null, passes, two_sided):
    """Add p_claimed_cal / p_opposite_cal: nominal when the statistic passes calibration, else empirical."""
    res = res.copy()
    if passes:
        res['p_claimed_cal'], res['p_opposite_cal'] = res.p_claimed, res.p_opposite
        return res
    if two_sided:
        emp = pl.calibrated(res.p.to_numpy(float), null)
        matches = np.isclose(res.p_claimed, res.p / 2)
        res['p_claimed_cal'] = np.where(matches, emp / 2, 1 - emp / 2)
        res['p_opposite_cal'] = np.where(matches, 1 - emp / 2, emp / 2)
    else:
        res['p_claimed_cal'] = pl.calibrated(res.p_claimed.to_numpy(float), null)
        res['p_opposite_cal'] = np.nan
    return res


def level_tests(level, programs, scopes, signs, calibration_key='level', passes_map=None, nulls=None, restrict=None):
    """Replication tests of programs at their scopes on the level t (condition and relabelings)."""
    passes_map = PASSES if passes_map is None else passes_map
    nulls = NULLS if nulls is None else nulls
    rows = []
    for scope in sorted(set(scopes.values())):
        progs = {p: programs[p] for p in programs if scopes[p] == scope}
        if not progs:
            continue
        stat = {part: level[(scope, part)].t for part in PARTS}
        if restrict is not None:
            stat = {part: s[s.index.isin(restrict)] for part, s in stat.items()}
        res = pl.program_test(stat['condition'], progs, {p: signs[p] for p in progs}, inter_gene_cor=CAMERA_COR)
        res = with_calibration(res, nulls[(calibration_key, scope)], passes_map[f'level t · {scope}'], True)
        for part in PARTS[1:]:
            rel = pl.program_test(stat[part], progs, {p: signs[p] for p in progs}, inter_gene_cor=CAMERA_COR)
            res[f'{part} p_claimed'] = rel.p_claimed
        res['scope'] = scope
        rows.append(res)
    return pd.concat(rows).reindex(list(programs))


def up_tests(statistics, programs, null, passes):
    """Up tests of programs on an F z (condition and relabelings)."""
    res = pl.program_test(statistics['condition'], programs, inter_gene_cor=CAMERA_COR)
    res = with_calibration(res, null, passes, False)
    for part in PARTS[1:]:
        res[f'{part} p_up'] = pl.program_test(statistics[part], programs, inter_gene_cor=CAMERA_COR).p_claimed
    return res


def conventional(programs, scopes, signs):
    rows = {}
    for p in programs:
        scope = scopes[p]
        row = {}
        for part in PARTS:
            stat = deseq_stat(part, scope)
            row[f'DESeq2 {part} p_claimed'] = float(pl.program_test(stat, {p: programs[p]}, {p: signs[p]},
                                                                    inter_gene_cor=CAMERA_COR).p_claimed.iloc[0])
        rows[p] = row
    return pd.DataFrame(rows).T


S_PROGS = {p: PROGRAMS_ALL[p] for p in CONFIRMATORY['species']}
S_SCOPE = {p: META.loc[p, 'segment'] for p in S_PROGS}
TEST_S = level_tests(LEVEL, S_PROGS, S_SCOPE, CLAIM_SIGN)
TEST_S['q_claimed'] = bh(TEST_S.p_claimed_cal)
TEST_S['q_opposite'] = bh(TEST_S.p_opposite_cal)
TEST_S['specific'] = (TEST_S['relabeling 1 p_claimed'] > ALPHA) & (TEST_S['relabeling 2 p_claimed'] > ALPHA)
TEST_S = TEST_S.join(conventional(S_PROGS, S_SCOPE, CLAIM_SIGN))


def between_tests(programs):
    seg = up_tests(Z_SEG, programs, NULLS['z_seg'], PASSES['segment F z (75)'])
    pos = up_tests(Z_POS, programs, NULLS['z_pos'], PASSES['positional F z (71)'])
    out = pd.DataFrame({'between p': seg.p_claimed_cal, 'between relabeling 1 p': seg['relabeling 1 p_up'],
                        'between relabeling 2 p': seg['relabeling 2 p_up'], 'axis p': pos.p_claimed_cal,
                        'axis relabeling 1 p': pos['relabeling 1 p_up'], 'axis relabeling 2 p': pos['relabeling 2 p_up']})
    out['between q'] = bh(out['between p'])
    out['axis q'] = bh(out['axis p'])
    return out


BETWEEN = {fam: between_tests({p: PROGRAMS_ALL[p] for p in CONFIRMATORY[fam]}) for fam in CONFIRMATORY}

Z_PROGS = {p: PROGRAMS_ALL[p] for p in CONFIRMATORY['zonation']}


def zonation_tests(programs, zon, adjust=True):
    rows = []
    for p, members in programs.items():
        seg = META.loc[p, 'segment']
        source = SOURCE_SPECIES[p]
        other = 'human' if source == 'mouse' else 'mouse'
        row = {'program': p, 'segment': seg, 'source species': source}
        for role, species in (('source', source), ('transfer', other)):
            res = pl.program_test(zon[(species, seg)].t, {p: members}, inter_gene_cor=CAMERA_COR)
            row[f'{role} p'] = float(res.p_claimed.iloc[0])
        rows.append(row)
    out = pd.DataFrame(rows).set_index('program')
    if adjust:
        out['source q'], out['transfer q'] = bh(out['source p']), bh(out['transfer p'])
    return out


TEST_Z = zonation_tests(Z_PROGS, ZON)
READING.append('The within-species zonation t has no relabeling (two specimens per species), so its program p is '
               'nominal (protocol: a positive control).')
display(TEST_S.round(4))
display(TEST_Z.round(4))
for fam in BETWEEN:
    display(BETWEEN[fam].round(4))

## 7 · Segment summaries and the decision rules (protocol 5)

**Summaries.** P_k is the median over members of the centred group difference in S1, S2 and S3, computed for the
condition, the 4 pairs and the 2 relabelings. Q_k = d · P_k:
- family S: d is the published sign;
- family Z: d is the sign of the condition P at the program's own segment.

**Classes.** Replication (5.1, family S), zonation and transfer (5.2, family Z) and localization (5.3) follow the
frozen rules.

In [ ]:
CONTRASTS = {'condition': pd.DataFrame({s: LEVEL[(s, 'condition')].effect for s in SCOPES}),
             **{part: pd.DataFrame({s: LEVEL[(s, part)].effect for s in SCOPES}) for part in PARTS[1:]},
             **{pair: pd.DataFrame({s: PAIR_CON[s][pair] for s in SCOPES}) for pair in PAIRS}}


def summarise(contrasts, programs, stat='median'):
    long = pl.segment_summary(contrasts, programs, min_members=MIN_MEMBERS, stat=stat)
    return long.pivot_table(index=['program', 'contrast'], columns='scope', values='value')


def zon_specimen_values(programs, zon, stat='median'):
    """Per program: the source and other species' per-specimen segment contrasts (program median)."""
    out = {}
    for p, members in programs.items():
        seg, source = META.loc[p, 'segment'], SOURCE_SPECIES[p]
        vals = {}
        for species, specimens in (('mouse', MOUSE), ('human', HUMAN)):
            frame = zon[(species, seg)]
            f = np.median if stat == 'median' else np.mean
            vals[species] = [float(f(frame.loc[members, s])) for s in specimens]
        out[p] = vals
    return out


def classify(programs, summary, between, family, test=None, zon_values=None, zon=None):
    rows = []
    for p in programs:
        cond = summary.loc[(p, 'condition'), list(SEGMENTS)].to_numpy(float)
        if family == 'species':
            d = CLAIM_SIGN[p]
        else:
            own = SEGMENTS.index(META.loc[p, 'segment'])
            d = 1 if cond[own] >= 0 else -1
        q_pairs = np.array([summary.loc[(p, pair), list(SEGMENTS)].to_numpy(float) for pair in PAIRS]) * d
        q_relab = np.array([summary.loc[(p, part), list(SEGMENTS)].to_numpy(float) for part in PARTS[1:]]) * d
        loc = pl.localization_class(between.loc[p, 'between q'], cond * d, q_pairs, q_relab)
        row = {'program': p, 'family': family, 'd': d, **{f'P {s}': cond[k] for k, s in enumerate(SEGMENTS)},
               'localization': loc['class'], 'peak': loc['peak'], 'margin': loc['margin'], 'pairs unanimous': loc['unanimous'],
               'localization specific': loc['specific'], 'relabeled ratio 1': loc['ratios'][0], 'relabeled ratio 2': loc['ratios'][1],
               'present in': ', '.join(loc['present']), 'no difference resolved in': ', '.join(loc['unresolved']),
               'between q': between.loc[p, 'between q'], 'axis q': between.loc[p, 'axis q']}
        if family == 'species':
            scope = S_SCOPE[p]
            at = summary.loc[(p, 'condition'), scope] * d
            pairs_at = np.array([summary.loc[(p, pair), scope] for pair in PAIRS]) * d
            row.update({'scope': scope, 'Q at scope': at, 'pairs with claimed sign': int((pairs_at > 0).sum()),
                        'q claimed': test.loc[p, 'q_claimed'], 'q opposite': test.loc[p, 'q_opposite'],
                        'specific (relabeled p > 0.05)': bool(test.loc[p, 'specific']),
                        'replication': pl.replication_class(test.loc[p, 'q_claimed'], bool(test.loc[p, 'specific']), at,
                                                            pairs_at, test.loc[p, 'q_opposite'])})
        else:
            seg, source = META.loc[p, 'segment'], SOURCE_SPECIES[p]
            other = 'human' if source == 'mouse' else 'mouse'
            vals = zon_values[p]
            zon = ZON if zon is None else zon
            x, y = zon[(source, seg)].effect.to_numpy(float), zon[(other, seg)].effect.to_numpy(float)
            row.update({'program ratio (other / source)': float(np.mean(vals[other]) / np.mean(vals[source])),
                        'genome-wide slope (other on source)': float((x * y).sum() / (x * x).sum())})
            row.update({'scope': seg, 'source species': source,
                        'source contrasts': vals[source], 'other contrasts': vals[other],
                        'source q': test.loc[p, 'source q'], 'transfer q': test.loc[p, 'transfer q'],
                        'replication': pl.transfer_class(test.loc[p, 'source q'], vals[source]),
                        'transfer': pl.transfer_class(test.loc[p, 'transfer q'], vals[other])})
        row['axis flag'] = ('changes along the axis but not between segments (descriptive, C14)'
                            if row['axis q'] <= ALPHA and row['between q'] > ALPHA else '')
        rows.append(row)
    out = pd.DataFrame(rows).set_index('program')
    out['informative'] = [pl.informative(r.replication if family == 'species' else
                                         ('replicated' if r.replication == 'zonated' else 'not replicated'), r.localization)
                          for r in out.itertuples()]
    return out


SUMMARY = summarise(CONTRASTS, {**S_PROGS, **Z_PROGS})
ZON_VALUES = zon_specimen_values(Z_PROGS, ZON)
CLASSES = pd.concat([classify(S_PROGS, SUMMARY, BETWEEN['species'], 'species', TEST_S),
                     classify(Z_PROGS, SUMMARY, BETWEEN['zonation'], 'zonation', TEST_Z, ZON_VALUES)])
display(CLASSES.drop(columns=['source contrasts', 'other contrasts'], errors='ignore').round(3))

## 8 · Program curves along the axis (coordinate version; display and sensitivity)

Notebook 76's rule:
- Δ(x) is centred by the median gene at each grid point (`centre_rows`).
- The program curve is the median over usable members (converged and not separated), in log2, signed by d.
- The pair curves are each human specimen curve minus each mouse specimen curve, with each pair's own convergence.
- The relabeled curves are the relabelings' Δ(x).
- Segment units come from 72's pooled transitions.
- **Axis agrees** when the floor of the curve's peak equals the label-based peak segment.
- Within-segment change is descriptive (C14).

In [ ]:
from pseudospace.aki_along_pt import centre_rows, profile_summary, program_profile


def curve_contrasts(models):
    cond = models['condition']
    ok = cond['stat|converged'].astype(bool) & ~cond['stat|separated'].astype(bool)
    out = {'condition': (centre_rows(cond['delta'], ok)[0], ok)}
    for part in PARTS[1:]:
        m = models[part]
        ok_r = m['stat|converged'].astype(bool) & ~m['stat|separated'].astype(bool)
        out[part] = (centre_rows(m['delta'], ok_r)[0], ok_r)
    for c in CASE:
        for r in REF:
            ok_p = (ok & cond[f'converged|{c}'].astype(bool) & cond[f'converged|{r}'].astype(bool)
                    & ~cond[f'separated|{c}'].astype(bool) & ~cond[f'separated|{r}'].astype(bool))
            diff = cond[f'curve|{c}'] - cond[f'curve|{r}']
            ref_ok = ok_p & np.isfinite(diff).all(axis=1)
            centred = diff - np.nanmedian(diff[ref_ok], axis=0)
            out[f'{c}−{r}'] = (centred, ok_p)
    return out


CURVE_CON = curve_contrasts(MODELS)


def program_curves(programs, signs, stat='median'):
    curves, rows = {}, []
    for p, members in programs.items():
        d = signs[p]
        for name, (centred, ok) in CURVE_CON.items():
            if stat == 'median':
                prof, n = program_profile(centred, GENES, members, usable=ok)
            else:
                idx = pd.Index(GENES).get_indexer(members)
                idx = idx[ok[idx]]
                prof, n = np.nanmean(centred[idx], axis=0) / np.log(2), len(idx)
            curves[(p, name)] = d * prof
            if name == 'condition':
                summ = profile_summary(d * prof, S_GRID)
                rows.append({'program': p, 'members used': n, **summ,
                             'axis peak segment': SEGMENTS[int(np.clip(np.floor(summ['peak']), 0, 2))]})
    return curves, pd.DataFrame(rows).set_index('program')


SIGNS = {p: int(CLASSES.loc[p, 'd']) for p in CLASSES.index}
CURVES, CURVE_SUMMARY = program_curves({**S_PROGS, **Z_PROGS}, SIGNS)
CLASSES['axis peak'] = CURVE_SUMMARY['axis peak segment']
CLASSES['axis'] = np.where(CLASSES['axis peak'] == CLASSES.peak, 'agrees', 'differs')
display(CURVE_SUMMARY.round(3))

## 9 · Sensitivities (protocol 4a and 7)

Each sensitivity reruns the relevant rule. A changed class is labelled beside the primary.
- **Equal probe counts:** members and background restricted to genes with equal probe counts; 5.1 rerun. A change is
  labelled "probe-sensitive".
- **Gene length:** the level t rank-normalised within deciles of log10 human genomic span; the replication test
  rerun. A change is labelled "length-sensitive". The between-segment test is shown stratified, descriptive only.
- **Mean instead of median** (O4): summaries and classes.
- **Untrimmed units:** all PT units, with no common-support trim.
- **Segment model check:** the peak of the median of 75's Δ_S against the pseudobulk peak.

In [ ]:
t0 = time.perf_counter()
SENS = pd.DataFrame(index=CLASSES.index)

# Equal probe counts (family S; also the zonation transfer is probe-free by construction, so S only)
eq_members = {p: [g for g in S_PROGS[p] if g in set(EQUAL_PROBE)] for p in S_PROGS}
testable = {p: v for p, v in eq_members.items() if len(v) >= MIN_MEMBERS}
SENS.loc[list(S_PROGS), 'equal-probe members'] = [len(eq_members[p]) for p in S_PROGS]
eq_nulls, eq_pass = {}, {}
for scope in SEGMENTS:
    row, eq_nulls[('level', scope)], _ = calibration_runs({p: LEVEL[(scope, p)].t[EQUAL_PROBE] for p in PARTS},
                                                           f'level t · {scope}', True)
    eq_pass[f'level t · {scope}'] = row['calibration'] == 'pass'
if testable:
    eq_test = level_tests(LEVEL, testable, {p: S_SCOPE[p] for p in testable}, CLAIM_SIGN, passes_map=eq_pass,
                          nulls=eq_nulls, restrict=set(EQUAL_PROBE))
    eq_test['q_claimed'], eq_test['q_opposite'] = bh(eq_test.p_claimed_cal), bh(eq_test.p_opposite_cal)
    eq_test['specific'] = (eq_test['relabeling 1 p_claimed'] > ALPHA) & (eq_test['relabeling 2 p_claimed'] > ALPHA)
    eq_summary = summarise(CONTRASTS, testable)
    for p in S_PROGS:
        if p not in testable:
            SENS.loc[p, 'equal-probe class'] = 'probe check not possible'
            continue
        d, scope = CLAIM_SIGN[p], S_SCOPE[p]
        at = eq_summary.loc[(p, 'condition'), scope] * d
        pairs_at = np.array([eq_summary.loc[(p, pair), scope] for pair in PAIRS]) * d
        SENS.loc[p, 'equal-probe class'] = pl.replication_class(eq_test.loc[p, 'q_claimed'], bool(eq_test.loc[p, 'specific']),
                                                                at, pairs_at, eq_test.loc[p, 'q_opposite'])
        SENS.loc[p, 'equal-probe q'] = eq_test.loc[p, 'q_claimed']

# Gene length (family S replication; between-segment descriptive)
len_level = {k: v.assign(t=pl.stratified_rank_normal(v.t, LENGTH_STRATA)) for k, v in LEVEL.items()}
len_nulls, len_pass = {}, {}
for scope in SEGMENTS:
    row, len_nulls[('level', scope)], _ = calibration_runs({p: len_level[(scope, p)].t.dropna() for p in PARTS},
                                                            f'level t · {scope}', True)
    len_pass[f'level t · {scope}'] = row['calibration'] == 'pass'
len_level = {k: v.dropna(subset=['t']) for k, v in len_level.items()}
len_test = level_tests(len_level, S_PROGS, S_SCOPE, CLAIM_SIGN, passes_map=len_pass, nulls=len_nulls)
len_test['q_claimed'], len_test['q_opposite'] = bh(len_test.p_claimed_cal), bh(len_test.p_opposite_cal)
len_test['specific'] = (len_test['relabeling 1 p_claimed'] > ALPHA) & (len_test['relabeling 2 p_claimed'] > ALPHA)
for p in S_PROGS:
    r = CLASSES.loc[p]
    pairs_at = np.array([SUMMARY.loc[(p, pair), r.scope] for pair in PAIRS]) * r.d
    SENS.loc[p, 'length class'] = pl.replication_class(len_test.loc[p, 'q_claimed'], bool(len_test.loc[p, 'specific']),
                                                       r['Q at scope'], pairs_at, len_test.loc[p, 'q_opposite'])
    SENS.loc[p, 'length q'] = len_test.loc[p, 'q_claimed']
    SENS.loc[p, 'members median span (kb)'] = float(SPAN[S_PROGS[p]].median() / 1e3)
SENS.loc[list(S_PROGS), 'universe median span (kb)'] = float(SPAN.median() / 1e3)
z_seg_len = {part: pl.stratified_rank_normal(Z_SEG[part], LENGTH_STRATA).dropna() for part in PARTS}
for fam in CONFIRMATORY:
    res = pl.program_test(z_seg_len['condition'], {p: PROGRAMS_ALL[p] for p in CONFIRMATORY[fam]}, inter_gene_cor=CAMERA_COR)
    SENS.loc[res.index, 'between p (length-stratified, descriptive)'] = res.p_claimed

# Mean instead of median
SUMMARY_MEAN = summarise(CONTRASTS, {**S_PROGS, **Z_PROGS}, stat='mean')
CLASSES_MEAN = pd.concat([classify(S_PROGS, SUMMARY_MEAN, BETWEEN['species'], 'species', TEST_S),
                          classify(Z_PROGS, SUMMARY_MEAN, BETWEEN['zonation'], 'zonation', TEST_Z,
                                   zon_specimen_values(Z_PROGS, ZON, 'mean'))])
SENS['mean: replication'] = CLASSES_MEAN.replication
SENS['mean: localization'] = CLASSES_MEAN.localization

# Untrimmed units (every exported PT unit)
all_counts = UNITS[:, GENES].X.tocsr()
measured = np.ones(UNITS.n_vars, bool) if 'measured_in_both' not in UNITS.var else UNITS.var.measured_in_both.astype(bool).to_numpy()
lib_cols = measured & ~UNITS.var_names.isin(MT)
all_library = np.asarray(UNITS.X[:, lib_cols].sum(axis=1)).ravel()
kept = UNITS.obs_names.get_indexer(INP['obs'].index)
np.testing.assert_allclose(all_library[kept], LIBRARY)                                        # the same library rule
all_specimen = UNITS.obs['sample'].astype(str).to_numpy()
all_segment = UNITS.obs.segment_class.astype(str).str.replace('PT-', '', regex=False).to_numpy()
RATES_ALL = pseudobulks(all_counts, all_library, all_specimen, all_segment)
LEVEL_ALL, PAIR_ALL = level_statistics(RATES_ALL, np.log(np.asarray(all_counts.mean(axis=0)).ravel() + 1e-300))
ZON_ALL = zonation_statistics(RATES_ALL, np.log(np.asarray(all_counts.mean(axis=0)).ravel() + 1e-300))
all_nulls, all_pass = {}, {}
for scope in SEGMENTS:
    row, all_nulls[('level', scope)], _ = calibration_runs({p: LEVEL_ALL[(scope, p)].t for p in PARTS}, f'level t · {scope}', True)
    all_pass[f'level t · {scope}'] = row['calibration'] == 'pass'
all_test = level_tests(LEVEL_ALL, S_PROGS, S_SCOPE, CLAIM_SIGN, passes_map=all_pass, nulls=all_nulls)
all_test['q_claimed'], all_test['q_opposite'] = bh(all_test.p_claimed_cal), bh(all_test.p_opposite_cal)
all_test['specific'] = (all_test['relabeling 1 p_claimed'] > ALPHA) & (all_test['relabeling 2 p_claimed'] > ALPHA)
CON_ALL = {'condition': pd.DataFrame({s: LEVEL_ALL[(s, 'condition')].effect for s in SCOPES}),
           **{part: pd.DataFrame({s: LEVEL_ALL[(s, part)].effect for s in SCOPES}) for part in PARTS[1:]},
           **{pair: pd.DataFrame({s: PAIR_ALL[s][pair] for s in SCOPES}) for pair in PAIRS}}
SUMMARY_ALL = summarise(CON_ALL, {**S_PROGS, **Z_PROGS})
TEST_Z_ALL = zonation_tests(Z_PROGS, ZON_ALL)
CLASSES_ALL = pd.concat([classify(S_PROGS, SUMMARY_ALL, BETWEEN['species'], 'species', all_test),
                         classify(Z_PROGS, SUMMARY_ALL, BETWEEN['zonation'], 'zonation', TEST_Z_ALL,
                                  zon_specimen_values(Z_PROGS, ZON_ALL), zon=ZON_ALL)])
SENS['untrimmed: replication'] = CLASSES_ALL.replication
SENS['untrimmed: localization'] = CLASSES_ALL.localization
if 'transfer' in CLASSES_ALL:
    SENS['untrimmed: transfer'] = CLASSES_ALL.transfer
READING.append('The untrimmed arm keeps the segment F-test q of the analysed units (it is a fitted statistic) and '
               'recomputes every pseudobulk quantity, the level t and its calibration on all exported PT units.')

# The segment model check: 75's Delta_S (natural log, centred by the median gene per segment)
delta75 = pd.DataFrame({s: SEG75['condition'][f'delta_{s}'] for s in SEGMENTS}).astype(float)
ok75 = SEG75['condition'].converged.astype(bool) & ~SEG75['condition'].separated.astype(bool)
delta75 = (delta75 - delta75[ok75].median()) / np.log(2)
for p in CLASSES.index:
    members = [g for g in PROGRAMS_ALL[p] if ok75.get(g, False)]
    vals = delta75.loc[members].median().to_numpy() * CLASSES.loc[p, 'd']
    SENS.loc[p, '75 Δ_S peak'] = SEGMENTS[int(np.argmax(vals))]
SENS['75 Δ_S peak agrees'] = SENS['75 Δ_S peak'] == CLASSES.peak

# Labels on the primary class
CLASSES['labels'] = ''
for p in S_PROGS:
    labels_p = []
    if SENS.loc[p, 'equal-probe class'] not in (CLASSES.loc[p, 'replication'], 'probe check not possible'):
        labels_p.append('probe-sensitive')
    if SENS.loc[p, 'length class'] != CLASSES.loc[p, 'replication']:
        labels_p.append('length-sensitive')
    CLASSES.loc[p, 'labels'] = ', '.join(labels_p)
display(SENS.round(3))
print(f'Sensitivities {time.perf_counter() - t0:.0f} s')

## 10 · Secondary programs (same analyses, nominal p, labelled secondary)

In [ ]:
SEC_PROGS = {p: PROGRAMS_ALL[p] for p in SECONDARY}
SEC_TEST = zonation_tests(SEC_PROGS, ZON, adjust=False)
SEC_TEST['source q'], SEC_TEST['transfer q'] = SEC_TEST['source p'], SEC_TEST['transfer p']    # nominal
sec_between = up_tests(Z_SEG, SEC_PROGS, NULLS['z_seg'], PASSES['segment F z (75)'])
sec_axis = up_tests(Z_POS, SEC_PROGS, NULLS['z_pos'], PASSES['positional F z (71)'])
SEC_BETWEEN = pd.DataFrame({'between q': sec_between.p_claimed_cal, 'axis q': sec_axis.p_claimed_cal})
SEC_SUMMARY = summarise(CONTRASTS, SEC_PROGS)
SEC_CLASSES = classify(SEC_PROGS, SEC_SUMMARY, SEC_BETWEEN, 'zonation', SEC_TEST, zon_specimen_values(SEC_PROGS, ZON))
SEC_CLASSES['p kind'] = 'nominal (secondary)'
SEC_CURVES, SEC_CURVE_SUMMARY = program_curves(SEC_PROGS, {p: int(SEC_CLASSES.loc[p, 'd']) for p in SEC_CLASSES.index})
SEC_CLASSES['axis peak'] = SEC_CURVE_SUMMARY['axis peak segment']
SEC_CLASSES['axis'] = np.where(SEC_CLASSES['axis peak'] == SEC_CLASSES.peak, 'agrees', 'differs')
display(SEC_CLASSES[['scope', 'source species', 'replication', 'transfer', 'localization', 'peak', 'between q']].round(3))

## 11 · Added information (protocol 4c; one templated line per program)

In [ ]:
FLAG_TEXT = {'S2': 'human S2 small (632 units)', 'S3': 'S3 compares human cortical S3 with mouse S3 including the outer stripe'}


def lines_for(classes, tests_q):
    out = {}
    for p, r in classes.iterrows():
        flags = []
        if r.peak in FLAG_TEXT:
            flags.append(FLAG_TEXT[r.peak])
        if r.family == 'species':
            flags.append('technology (CellSpectra: one species against two, single-nucleus human data)')
            if r.get('labels'):
                flags.append(r['labels'])
        row = {'source': str(META.loc[p, 'source']).split(' (')[0], 'claim': META.loc[p, 'published_claim'].split(' (')[0],
               'replication': r.replication, 'scope': r.scope, 'localization': r.localization, 'peak': r.peak,
               'unresolved': [s for s in str(r['no difference resolved in']).split(', ') if s], 'axis': r.axis,
               'flags': flags}
        if r.family == 'species':
            row.update(value=r['Q at scope'] * r.d, pairs=int(r['pairs with claimed sign']), q=tests_q[p])
        else:
            row['transfer'] = f"{r.transfer} (published {r['source species']} {r.scope} program)"
        out[p] = pl.added_information(row)
    return pd.Series(out, name='added information')


ADDED = pd.concat([lines_for(CLASSES, TEST_S.q_claimed.to_dict()), lines_for(SEC_CLASSES, {})])
for p, line in ADDED.items():
    print(f'{p}: {line}')

## 12 · Figures

- **Per family, small multiples** (protocol 4d). One row per program, three panels:
  1. the program curve on segment units: the condition thick, the 4 pairs thin, the relabelings dashed;
  2. the member heatmap trio: mouse mean, human mean and their difference, on 20 equal-count bins per specimen;
  3. segment bars: the condition, the 4 pairs as dots and the relabelings as crosses.
- **Summary heatmap:** programs × S1/S2/S3, coloured by Q (log2, clipped at ±2). Dots count the agreeing pairs; the
  box marks the localized segment, the underline the published segment, and the class sits at the row end.

In [ ]:
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle

ALL_RATES, BIN_MAJORITY, BIN_NAMES = pl.bin_rates(COUNTS, LIBRARY, POSITION, SPECIMEN, SEGMENT, np.arange(len(GENES)),
                                                  n_bins=N_BINS, floor=FLOOR)
spec_index = {n: i for i, n in enumerate(BIN_NAMES)}
ref_mean = np.mean([ALL_RATES[spec_index[s]] for s in REF], axis=0)          # bins x genes
case_mean = np.mean([ALL_RATES[spec_index[s]] for s in CASE], axis=0)
diff_bins = case_mean - ref_mean
diff_bins = diff_bins - np.median(diff_bins, axis=0)
gene_pos = {g: i for i, g in enumerate(GENES)}
REF_LABEL, CASE_LABEL = 'mouse', 'human'


def heat_rows(members):
    rows = members[:MAX_ROWS]
    idx = [gene_pos[g] for g in rows]
    order = np.argsort(np.argmax(ref_mean[:, idx], axis=0), kind='stable')
    return [rows[i] for i in order], [idx[i] for i in order]


def family_figure(programs, classes, title, stem):
    n = len(programs)
    fig, axes = plt.subplots(n, 5, figsize=(183 * MM, 34 * MM * n), gridspec_kw={'width_ratios': [1.25, .7, .7, .7, .75]},
                             squeeze=False)
    for i, p in enumerate(programs):
        r = classes.loc[p]
        ax = axes[i, 0]
        for part, style in (('relabeling 1', '--'), ('relabeling 2', ':')):
            ax.plot(S_GRID, CURVES[(p, part)], color=RELABEL_COLOR, lw=.7, ls=style)
        for pair in PAIRS:
            ax.plot(S_GRID, CURVES[(p, pair)], color=HUMAN_COLOR, lw=.5, alpha=.6)
        ax.plot(S_GRID, CURVES[(p, 'condition')], color=INK, lw=1.4)
        for b in (1, 2):
            ax.axvline(b, color=LIGHT, lw=.6, zorder=0)
        ax.axhline(0, color=MUTED, lw=.5, zorder=0)
        ax.set_xlim(S_GRID.min(), S_GRID.max())
        ax.set_ylabel(f'd·Δ (log2), d = {int(r.d):+d}')
        ax.set_title(f'{p} · {META.loc[p, "source"].split(" (")[0]} · {META.loc[p, "direction"]}', loc='left')
        if i == n - 1:
            ax.set_xlabel('segment units (S1 0–1, S2 1–2, S3 2–3; within-segment shape descriptive)')
        rows, idx = heat_rows(PROGRAMS_ALL[p])
        both = np.concatenate([ref_mean[:, idx], case_mean[:, idx]], axis=0)
        centred = both - both.mean(axis=0)
        lim = np.nanpercentile(np.abs(centred), 98) or 1
        for j, (mat, lab) in enumerate(((centred[:N_BINS], REF_LABEL), (centred[N_BINS:], CASE_LABEL), (diff_bins[:, idx], 'human − mouse'))):
            ax = axes[i, j + 1]
            vmax = lim if j < 2 else max(np.nanpercentile(np.abs(diff_bins[:, idx]), 98), .1)
            ax.imshow(mat.T, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax, interpolation='nearest')
            ax.set_yticks(range(len(rows)))
            ax.set_yticklabels(rows if j == 0 else [], fontsize=3.2)
            ax.set_xticks([])
            group = REF if j == 0 else CASE if j == 1 else REF
            maj = BIN_MAJORITY.loc[list(group)].mode().iloc[0]
            for b in range(N_BINS):
                ax.add_patch(Rectangle((b - .5, len(rows) - .5), 1, max(len(rows) * .04, .6), clip_on=False,
                                       color={'S1': '#cfd8dc', 'S2': '#90a4ae', 'S3': '#455a64'}[maj[b]], lw=0))
            ax.set_title(f'{lab} ({len(rows)} of {len(PROGRAMS_ALL[p])})', fontsize=5.5, fontweight='normal')
        ax = axes[i, 4]
        x = np.arange(3)
        ax.bar(x, [r[f'P {s}'] for s in SEGMENTS], color=[LIGHT] * 3, edgecolor=INK, lw=.5)
        for k, s in enumerate(SEGMENTS):
            ax.scatter([k] * 4, [SUMMARY_FOR[p].loc[pair, s] for pair in PAIRS], s=4, color=HUMAN_COLOR, zorder=3)
            ax.scatter([k] * 2, [SUMMARY_FOR[p].loc[part, s] for part in PARTS[1:]], s=8, marker='x', color=RELABEL_COLOR, zorder=3)
        pub = SEGMENTS.index(META.loc[p, 'segment'])
        ax.add_patch(Rectangle((pub - .45, ax.get_ylim()[0]), .9, np.ptp(ax.get_ylim()), fill=False, ec=HUMAN_COLOR, lw=.8, ls='--'))
        ax.axhline(0, color=MUTED, lw=.5)
        ax.set_xticks(x)
        ax.set_xticklabels(SEGMENTS)
        ax.set_ylabel('human − mouse (log2)')
        rep = r.replication + (f' · transfer {r.transfer}' if 'transfer' in r and isinstance(r.get('transfer'), str) else '')
        ax.set_title(f'{rep}\n{r.localization}', fontsize=5.5, fontweight='normal')
    fig.suptitle(title, x=0, ha='left', fontweight='bold')
    fig.tight_layout()
    save(fig, stem)


SUMMARY_FOR = {p: SUMMARY.xs(p, level='program') for p in {**S_PROGS, **Z_PROGS}}
family_figure(CONFIRMATORY['species'], CLASSES, 'Species family S: CellSpectra human-high / mouse-high blocks', 'fig79a_species_programs')
family_figure(CONFIRMATORY['zonation'], CLASSES, 'Zonation family Z: mouse (Chen 2021) and human (Hinze 2022) segment programs',
              'fig79b_zonation_programs')


def summary_heatmap(classes, summary, stem, title):
    progs = list(classes.index)
    q = np.array([[summary.loc[(p, 'condition'), s] * classes.loc[p, 'd'] for s in SEGMENTS] for p in progs])
    fig, ax = plt.subplots(figsize=(150 * MM, (22 + 5.5 * len(progs)) * MM))
    im = ax.imshow(np.clip(q, -2, 2), cmap='RdBu_r', norm=TwoSlopeNorm(0, -2, 2), aspect='auto')
    for i, p in enumerate(progs):
        r = classes.loc[p]
        for k, s in enumerate(SEGMENTS):
            agree = sum(np.sign(summary.loc[(p, pair), s] * r.d) == np.sign(q[i, k]) for pair in PAIRS)
            ax.text(k, i, '•' * int(agree), ha='center', va='center', fontsize=5, color=INK)
        if str(r.localization).startswith('localized'):
            k = SEGMENTS.index(r.peak)
            ax.add_patch(Rectangle((k - .5, i - .5), 1, 1, fill=False, ec=INK, lw=1.4))
        pub = SEGMENTS.index(META.loc[p, 'segment'])
        ax.plot([pub - .4, pub + .4], [i + .42, i + .42], color=HUMAN_COLOR, lw=1.2)
        rep = r.replication + (f' / {r.transfer}' if isinstance(r.get('transfer'), str) else '')
        ax.text(2.6, i, f'{rep} · {r.localization}' + (f" · {r['labels']}" if r.get('labels') else ''), va='center', fontsize=5.5)
    ax.set_xticks(range(3))
    ax.set_xticklabels(SEGMENTS)
    ax.set_yticks(range(len(progs)))
    ax.set_yticklabels([f'{p} ({META.loc[p, "direction"]})' for p in progs])
    ax.set_xlim(-.5, 2.5)
    fig.colorbar(im, ax=ax, orientation='horizontal', fraction=.04, pad=.12, aspect=30,
                 label='Q = d · median centred difference (log2); d = published (S) or own-segment (Z) sign')
    ax.set_title(title, loc='left')
    save(fig, stem)


summary_heatmap(CLASSES, SUMMARY, 'fig79c_summary_heatmap', 'Published programs × segment (dots: pairs agreeing; box: localized; line: published segment)')
summary_heatmap(SEC_CLASSES, SEC_SUMMARY, 'supp79_secondary_heatmap', 'Secondary zonation programs (nominal p)')

## 13 · Outputs

In [ ]:
META.to_csv(tables / 'programs.csv')
MEMBERS_TABLE.to_csv(tables / 'programs_members.csv', index=False)
pd.concat({s: pd.DataFrame(RATES[s]) for s in SCOPES}, axis=1).to_csv(tables / 'pseudobulk_rates.csv.gz')
gene_rows = []
for p, members in {**S_PROGS, **Z_PROGS, **SEC_PROGS}.items():
    for g in members:
        row = {'program': p, 'gene': g, 'z_seg': Z_SEG['condition'][g], 'z_pos': Z_POS['condition'][g],
               'converged (71)': bool(STAT71['condition'].converged[g]), 'separated (71)': bool(STAT71['condition'].separated[g]),
               'equal probes': g in set(EQUAL_PROBE), 'span (kb)': SPAN[g] / 1e3}
        for s in SCOPES:
            row[f'condition {s}'] = LEVEL[(s, 'condition')].effect[g]
            row[f't {s}'] = LEVEL[(s, 'condition')].t[g]
            for pair in PAIRS:
                row[f'{pair} {s}'] = PAIR_CON[s][pair][g]
        gene_rows.append(row)
GENE_LEVEL = pd.DataFrame(gene_rows).merge(MEMBERS_TABLE.rename(columns={'program_id': 'program'}), on=['program', 'gene'], how='left')
GENE_LEVEL.to_csv(tables / 'gene_level.csv', index=False)
share = []
for p in S_PROGS:
    d, scope = CLAIM_SIGN[p], S_SCOPE[p]
    vals = CONTRASTS['condition'].loc[S_PROGS[p], scope] * d
    pairs_ok = np.all([CONTRASTS[pair].loc[S_PROGS[p], scope] * d > 0 for pair in PAIRS], axis=0)
    base = CONTRASTS['condition'][scope] * d
    base_pairs = np.all([CONTRASTS[pair][scope] * d > 0 for pair in PAIRS], axis=0)
    share.append({'program': p, 'scope': scope, 'share claimed sign': float((vals > 0).mean()),
                  'share claimed sign, all 4 pairs': float(pairs_ok.mean()), 'universe share': float((base > 0).mean()),
                  'universe share, all 4 pairs': float(base_pairs.mean())})
GENE_SHARES = pd.DataFrame(share).set_index('program')
display(GENE_SHARES.round(3))
PROGRAM_TESTS = pd.concat({'species': TEST_S.join(BETWEEN['species']), 'zonation': TEST_Z.join(BETWEEN['zonation'])},
                          names=['family'])
PROGRAM_TESTS.to_csv(tables / 'program_tests.csv')
pd.concat({'median': SUMMARY, 'mean': SUMMARY_MEAN, 'untrimmed (median)': SUMMARY_ALL}, names=['statistic']).to_csv(
    tables / 'segment_summary.csv')
np.savez_compressed(tables / 'profiles.npz', grid=GRID, s=S_GRID,
                    **{f'{p}|{name}': v for (p, name), v in CURVES.items()})
CURVE_SUMMARY.to_csv(tables / 'profile_summaries.csv')
CLASSES.join(GENE_SHARES.drop(columns='scope')).to_csv(tables / 'classes.csv')
SEC_CLASSES.to_csv(tables / 'classes_secondary.csv')
ADDED.to_csv(tables / 'added_information.csv')
SENS.to_csv(tables / 'sensitivity.csv')
CLASSES_MEAN.to_csv(tables / 'sensitivity_mean_classes.csv')
CLASSES_ALL.to_csv(tables / 'sensitivity_untrimmed_classes.csv')

## 14 · Readout (protocol 5.4) and the abandonment rule

In [ ]:
conf = CLASSES
counts_info = conf.groupby('family').informative.value_counts().unstack(fill_value=0)
display(counts_info)
resolved = conf.localization.str.startswith('localized') | conf.localization.isin(['graded', 'uniform within resolution'])
n_resolved_s = int(resolved[conf.family.eq('species')].sum())
outside = [p for p in conf.index if str(conf.loc[p, 'localization']).startswith('localized') and conf.loc[p, 'peak'] != META.loc[p, 'segment']]
ABANDON = n_resolved_s < 2
print(f'Species family: {n_resolved_s} of 6 programs reach a resolved localization class (abandonment rule: fewer than 2) → '
      + ('position adds no resolvable information to the species claims' if ABANDON else 'the rule is not met'))
print('Localized programs whose peak lies outside the published segment:', outside or 'none')
display(conf[['family', 'scope', 'replication', 'transfer', 'localization', 'peak', 'axis', 'labels', 'informative']]
        if 'transfer' in conf else conf)

## 15 · How the protocol was read

In [ ]:
READING.extend([
    'Programs are read from the committed docs/results/known-programs/ files (relative to the repository), with sha256 asserted.',
    "Relabeled specificity of a replication test uses the program's relabeled one-sided p in the claimed direction, as "
    'the protocol writes; the relabeled p are nominal.',
    'Family Z: the transfer class (5.2) is evaluated with the transfer q (BH over the 6 programs); the localization d is '
    "the sign of the condition P at the program's own segment (+1 when it is 0).",
    "Transfer classes are evaluated in the order zonated, reversed, absent (signs differ or both within ±0.25), "
    'attenuated (both positive); a remaining case (both negative, not both ≤ −0.25) is read as absent.',
    'Heatmap rows are the first 40 members in the paper\'s ranking order (data-independent), ordered by the mouse peak bin.',
    "The summary heatmap colours Q = d·P: the published direction for family S, the observed own-segment sign for family Z.",
    'The pair curves are centred by the pair\'s own median gene at each grid point over genes finite there (76 centres '
    'with its convergence mask; NaN outside a specimen support is excluded).',
    'The covariate of the untrimmed arm is its own log mean count per unit.',
    'Notebook 79 fits no model, so the stage cache stays empty; the calibration runs (about 40 cameraPR calls) are recomputed.',
])
HOW = pd.Series(READING, name='how the protocol was read')
HOW.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
for line in READING:
    print('·', line)

## 16 · Run record

In [ ]:
record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'protocol': 'docs/results/pt-known-programs.md (commit b90a682)',
          'pass1_fingerprint': FINGERPRINT, 'program_sha256': PROGRAM_SHA, 'ensembl_sha256': ENSEMBL_SHA,
          'units': int(len(POSITION)), 'genes': len(GENES), 'library_sets': len(SETS),
          'guards': GUARDS, 'calibration': CALIBRATION.reset_index().to_dict('records'),
          'abandonment_rule': {'species resolved': n_resolved_s, 'met': bool(ABANDON)},
          'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(record, indent=1, default=str))
print(f"Done in {record['seconds']:.0f} s")